### Part 1: Cohort Analysis & Retention Analytics

##### Question 1 (First-Time vs Repeat Customers): 
Identify customer purchasing behavior by calculating the total count of customers who have placed exactly 1 order versus those who placed 2 or more orders in olist_orders_dataset.

In [0]:
with customer_total_orders as (
select customer_unique_id,count(order_id)  total_order
from brazilian_e_commerce.sql_practice.olist_orders_dataset ord 
join brazilian_e_commerce.sql_practice.olist_customers_dataset cus on 
ord.customer_id=cus.customer_id
group by customer_unique_id)
select 
case
when total_order = 1 then '1 Order'
else '2 or more'
end as customer_type,
count(customer_unique_id) as total_customer
from customer_total_orders
group by customer_type;

In [0]:
with customer_total_order as (
    select customer_unique_id,count(order_id)  total_order
from brazilian_e_commerce.sql_practice.olist_orders_dataset ord 
join brazilian_e_commerce.sql_practice.olist_customers_dataset cus on 
ord.customer_id=cus.customer_id 
group by customer_unique_id)
select count(case when total_order = 1 then 1 end) as single_order_customer,
count(case when total_order > 1 then 1 end) as multiple_order_customer
from customer_total_order;

##### Question 2 (Customer Cohort Month): 
For each customer (customer_unique_id via joining olist_customers_dataset), determine their cohort month (their very first purchase month). Group customers by cohort month and track total unique customers per cohort.

In [0]:
with customer_cohort as (
select cus.customer_unique_id,date_trunc('month',min(ord.order_purchase_timestamp)) cohort_month
from brazilian_e_commerce.sql_practice.olist_orders_dataset ord
join brazilian_e_commerce.sql_practice.olist_customers_dataset cus on 
ord.customer_id=cus.customer_id
group by cus.customer_unique_id)
select cohort_month,count(customer_unique_id) as total_customer
from customer_cohort
group by cohort_month
order by total_customer desc;

##### Question 3 (Customer Lifetime Value - LTV Quartiles):

Calculate the total lifetime spend per customer_unique_id (summing payment_value from olist_order_payments_dataset across their orders), and rank customers into spend quartiles using NTILE(4).

In [0]:
select customer_unique_id,sum(payment_value) as total_spend,
ntile(4) over(order by sum(payment_value) desc) as spend_quartile
from brazilian_e_commerce.sql_practice.olist_customers_dataset cus
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on cus.customer_id=ord.customer_id
join brazilian_e_commerce.sql_practice.olist_order_payments_dataset pay on ord.order_id=pay.order_id
group by customer_unique_id
order by total_spend desc;

### Part 2: Funnel, Performance & Lead Time Engineering

##### Question 4 (SLA / Late Delivery Analysis): 
Identify orders where order_delivered_customer_date exceeded order_estimated_delivery_date. Calculate the total count and percentage of late orders across the entire platform.

In [0]:
select count(order_id) tot_count_late_ord,
round(
    (count(order_id)*100.0)/(select count(order_id) from brazilian_e_commerce.sql_practice.olist_orders_dataset) ,2) as pct_late_ord
from brazilian_e_commerce.sql_practice.olist_orders_dataset
where order_delivered_customer_date > order_estimated_delivery_date;

##### Question 5 (Fulfillment Bottleneck Analysis): Break down the total fulfillment duration into two stages for delivered orders:

1.Approval delay: DATEDIFF(order_approved_at, order_purchase_timestamp)

2.Carrier shipping delay: DATEDIFF(order_delivered_customer_date, order_delivered_carrier_date)
Return the average days for both stages per year-month.

In [0]:
select date_trunc('month',order_purchase_timestamp) as month,
round(avg(date_diff(order_approved_at,order_purchase_timestamp)),2) approval_delay,
round(avg(date_diff(order_delivered_customer_date,order_delivered_carrier_date)),2) carrier_delay
from brazilian_e_commerce.sql_practice.olist_orders_dataset 
where order_status = 'delivered'
group by date_trunc('month',order_purchase_timestamp)
order by month;

##### Question 6 (Order Status Funnel Drop-off):

Calculate the absolute count of orders and the percentage breakdown for each order_status across the entire olist_orders_dataset. Order the results by total orders descending.

In [0]:
select order_status,count(order_id) order_count,
round(count(order_id)*100.0/(select count(order_id) from brazilian_e_commerce.sql_practice.olist_orders_dataset),2) pct_order
from brazilian_e_commerce.sql_practice.olist_orders_dataset
group by order_status
order by order_count desc;

### Part 3: Advanced Windows, Gaps, & Trends

##### Question 7 (Time Between Consecutive Orders):
For customers with multiple orders, calculate the gap in days between each order and their previous order using LAG()

In [0]:
with order_gaps as (
select cus.customer_unique_id,
date_diff(ord.order_purchase_timestamp, lag(ord.order_purchase_timestamp) over (partition by cus.customer_unique_id order by ord.order_purchase_timestamp)) as date_dif
from brazilian_e_commerce.sql_practice.olist_customers_dataset cus 
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on
cus.customer_id=ord.customer_id
)
select * from order_gaps where date_dif is not null;

##### Question 8 (3-Month Rolling Average Revenue):

Calculate total revenue (SUM(payment_value)) per month, then compute a 3-month rolling average revenue using frame specifications (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW).

In [0]:
with monthly_revenue as (
select date_trunc('month',ord.order_purchase_timestamp) month,round(sum(ordpay.payment_value),2) as total_revenue
from brazilian_e_commerce.sql_practice.olist_order_payments_dataset ordpay
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on ordpay.order_id=ord.order_id
group by date_trunc('month',order_purchase_timestamp))
select month,total_revenue,round(avg(total_revenue) over (order by month rows between 2 preceding and current row),2) as 3_month_rol_avg
from monthly_revenue
order by month;


##### Question 9 (Month-over-Month Revenue Growth %):

Compute total payment revenue per month, fetch the prior month's revenue using LAG(), and calculate the MoM growth percentage:


In [0]:
with monthly_revenue as (
select date_trunc('month',ord.order_purchase_timestamp) month,
round(sum(ordpay.payment_value),2) total_revenue
from brazilian_e_commerce.sql_practice.olist_orders_dataset ord 
join brazilian_e_commerce.sql_practice.olist_order_payments_dataset ordpay
on ord.order_id=ordpay.order_id
group by date_trunc('month',ord.order_purchase_timestamp)
)
select month, total_revenue,((total_revenue-round(lag(total_revenue,1) over(order by month),2))/
round(lag(total_revenue,1) over(order by month),2))*100 as mom_revenue
from monthly_revenue
order by month;


### Part 4?
##### Question 10 (Payment vs Order Item Discrepancy):
Aggregate total item price + freight (SUM(price + freight_value)) per order_id from olist_order_items_dataset, and aggregate total payment_value (SUM(payment_value)) per order_id from olist_order_payments_dataset. Find any orders where the payment sum does not equal the item/freight sum.

In [0]:
with item_total_per_order as (
    select order_id,round(sum(price+ freight_value),2) as item_total_price
    from brazilian_e_commerce.sql_practice.olist_order_items_dataset
    group by order_id
),
payment_total as (
    select order_id,round(sum(payment_value),2) as payment_total
    from brazilian_e_commerce.sql_practice.olist_order_payments_dataset
    group by order_id
)
select ordtot.order_id,ordtot.item_total_price,paytot.payment_total
from item_total_per_order ordtot
join payment_total paytot
on ordtot.order_id=paytot.order_id
where item_total_price != payment_total; 


##### Question 11 (Multi-Payment Type Breakdown):

Identify orders that used more than 1 distinct payment method (payment_type). Display order_id, the count of distinct payment methods used, and concatenate the distinct payment methods into a single string (using COLLECT_SET() + ARRAY_JOIN() or CONCAT_WS()).

In [0]:
select order_id,count(distinct payment_type) as distict_payment_types_count,
array_join(collect_set(payment_type),',') as payment_methods
from brazilian_e_commerce.sql_practice.olist_order_payments_dataset
group by order_id
having count(distinct payment_type)>1;

##### Question 12 (Top Product Category per State): 
Determine the single highest-grossing product category (SUM(price)) for each customer state (customer_state). Output customer_state, product_category_name, and total sales using ROW_NUMBER().

In [0]:
with customer_state_sales as (
select cust.customer_state,prod.product_category_name,sum(orditm.price) as total_sales
from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on orditm.order_id=ord.order_id
join brazilian_e_commerce.sql_practice.olist_products_dataset prod on orditm.product_id=prod.product_id
join brazilian_e_commerce.sql_practice.olist_customers_dataset cust on ord.customer_id=cust.customer_id
group by cust.customer_state,prod.product_category_name
),
customer_state_sales_rank as(
select customer_state,product_category_name,total_sales,rank() over (partition by customer_state order by total_sales desc) as sales_rank
from customer_state_sales
)
select * from customer_state_sales_rank where sales_rank=1;


##### Question 13 (Pareto 80/20 Rule on Sellers): 
Calculate total revenue per seller. Rank sellers by revenue in descending order and compute a running percentage of total platform revenue to find the top sellers that generate 80% of total revenue.

In [0]:
with seller_revenue_rank as (
    select seller_id,round(sum(price),2) as revenue
from brazilian_e_commerce.sql_practice.olist_order_items_dataset
group by seller_id),
seller_running_revenue as (
select seller_id,revenue,
dense_rank() over (order by revenue desc) as seller_rank,
round(
    sum(revenue) over (order by revenue desc rows between unbounded preceding and current row)*100.0/(select sum(price) from brazilian_e_commerce.sql_practice.olist_order_items_dataset),2) as running_pct
from seller_revenue_rank)
select seller_id,revenue,seller_rank,running_pct 
from seller_running_revenue
where running_pct<= 80.0
order by revenue desc;

##### Question 14 (Seller Recency, Frequency, Monetary - RFM Segment): 
Build an RFM model per seller:
Recency: Days since last order item sold.
Frequency: Total unique orders fulfilled.
Monetary: Total revenue generated.

In [0]:
select seller_id,date_diff(
    (select max(shipping_limit_date) from brazilian_e_commerce.sql_practice.olist_order_items_dataset),max(shipping_limit_date)
) Recency,
count(distinct order_id) frequency,
round(sum(price),2) Monetary_revenue
from brazilian_e_commerce.sql_practice.olist_order_items_dataset
group by seller_id
order by Monetary_revenue desc;

##### Question 15 (Churned Sellers Analysis):

Define a seller as "churned" if they made sales in 2017 but have zero sales in 2018. Return the total count of churned sellers and their total 2017 revenue loss.

In [0]:
with sellersales_revenue as (
select seller_id,
count(case when year(shipping_limit_date) = 2017 then 1 end) as sales_2017,
count(case when year(shipping_limit_date) = 2018 then 1 end) as sales_2018,
round(sum(case when year(shipping_limit_date) = 2017 then price else 0 end),2) as rev_2017
from brazilian_e_commerce.sql_practice.olist_order_items_dataset
group by seller_id)
select count(seller_id) as chured_seller_count,round(sum(rev_2017),2) as lost_revenue_2017 
from sellersales_revenue
where sales_2017 > 0 and sales_2018 = 0;